In [46]:
import pandas as pd

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 39073 entries, 0 to 39072
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   sampleid             39073 non-null  int64
 1   age                  39073 non-null  int64
 2   workclass            36849 non-null  str  
 3   fnlwgt               39073 non-null  int64
 4   education            39073 non-null  str  
 5   educational_num      39073 non-null  int64
 6   marital_status       39073 non-null  str  
 7   occupation           36839 non-null  str  
 8   relationship         39073 non-null  str  
 9   race                 39073 non-null  str  
 10  gender               39073 non-null  str  
 11  capital_gain         39073 non-null  int64
 12  capital_loss         39073 non-null  int64
 13  hours_per_week       39073 non-null  int64
 14  native_country       38392 non-null  str  
 15  income               39073 non-null  str  
 16  profile_description  39073 non-nu

In [47]:
train['income'].unique()

<StringArray>
['<=50K', '>50K']
Length: 2, dtype: str

In [48]:
train['income'] = (train['income'] == '>50K').astype(int)
train['income'].sum()

np.int64(9349)

In [49]:
no_us = train[train['native_country'] != 'United-States'].dropna(subset=['native_country'])
counts = no_us.groupby('native_country')['income'].sum()
answer_1 = counts.idxmax()
print(isinstance(answer_1, str))

True


In [50]:
rates = train.groupby('occupation')['income'].mean()
answer_2 = rates.idxmax()

In [51]:
import numpy as np

target = 'income'
drop_cols = ['sampleid', 'education', 'profile_description']

numeric_cols = ['age', 'fnlwgt', 'educational_num', 'capital_gain', 'capital_loss', 'hours_per_week']
categorical_cols = ['workclass', 'marital_status', 'occupation', 'relationship', 'race', 'gender', 'native_country', 'profile_description']

X = train[numeric_cols + categorical_cols]
y = train[target]

In [52]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric_steps = Pipeline([
	('impute', SimpleImputer(strategy='median')),
	('scale', StandardScaler()),
])

categorical_steps = Pipeline([
	('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
	('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocess = ColumnTransformer([
	('num', numeric_steps, numeric_cols),
	('cat', categorical_steps, categorical_cols),
])

model = Pipeline([
	('pre', preprocess),
	('model', LogisticRegression(max_iter=2000))
])

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)
oof = cross_val_predict(model, X, y, cv=skf, method='predict_proba')[:, 1]

thresholds = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(y, oof >= t, average='macro') for t in thresholds]
best = int(np.argmax(f1s))

33


In [56]:
print(round(f1s[best], 4), thresholds[best])

0.7922 0.37999999999999995


In [55]:
threshold = thresholds[best]
model.fit(X, y)

X_test = test[numeric_cols + categorical_cols]
proba_test = model.predict_proba(X_test)[:, 1]
predict_test = (proba_test >= threshold).astype(int)

print(predict_test.mean())

0.24966731497594433


In [58]:
from lightgbm import LGBMClassifier

def prepare_lgbm(df, reference=None):
	out = df[numeric_cols + categorical_cols].copy()
	for col in categorical_cols:
		values = out[col].fillna('Unknown')
		if reference is None:
			out[col] = values.astype('category')
		else:
			out[col] = pd.Categorical(values, categories=reference[col].cat.categories)
	return out

X_lgb = prepare_lgbm(train)
X_test_lgb = prepare_lgbm(test, reference=X_lgb)

/tmp/ipykernel_2225258/3568542766.py:10: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  out[col] = pd.Categorical(values, categories=reference[col].cat.categories)


In [59]:
lgb_model = LGBMClassifier(
	n_estimators=300,
	learning_rate=0.05,
	random_state=67,
	verbose=-1
)

In [62]:
oof_lgb = cross_val_predict(lgb_model, X_lgb, y, cv=skf, method='predict_proba')[:, 1]
f1s_lgb = [f1_score(y, oof_lgb >= t, average='macro') for t in thresholds]
best_lgb = int(np.argmax(f1s_lgb))
print("Best score for LightGBM is ", round(f1s_lgb[best_lgb], 4), "at threshold ", thresholds[best_lgb])
print('Best score for Logistic Regression is ', round(f1s[best], 4), "at threshold ", thresholds[best])

Best score for LightGBM is  0.8201 at threshold  0.42999999999999994
Best score for Logistic Regression is  0.7922 at threshold  0.37999999999999995
